# Hybrid Multi-Stage Attention + Clinical Prior Guided Framework

This notebook implements the novel pipeline for Chest X-ray classification:
1. **CLAHE Pre-processing**: For better contrast.
2. **Hybrid Backbone**: Convolutional (DenseNet121) + Vision Transformer (Swin-Tiny).
3. **Dual Attention**: Spatial and Channel attention modules.
4. **Focal Loss**: For handling class imbalance.
5. **Validation**: Per-class AUC metrics.

In [13]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from PIL import Image
from tqdm.auto import tqdm
import torch.nn.functional as F

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

Using device: mps


In [14]:
DATA_DIR = 'data/images'
SUBSET_METADATA_PATH = 'data/metadata_subset_10_percent.csv'
TARGET_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 
    'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 2

## 1. Disease-Aware Pre-processing (CLAHE)

In [15]:
class NovelDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None, use_clahe=True):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform
        self.use_clahe = use_clahe
        if use_clahe:
            self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            img_name = self.dataframe.iloc[idx]['Image Index']
            img_path = os.path.join(self.root_dir, img_name)
            
            image = cv2.imread(img_path)
            if image is None:
                raise FileNotFoundError(f"Image not found: {img_path}")
                
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            if self.use_clahe:
                lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
                l, a, b = cv2.split(lab)
                cl = self.clahe.apply(l)
                limg = cv2.merge((cl, a, b))
                image = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
            
            image = Image.fromarray(image)
            label = torch.tensor(self.dataframe.iloc[idx]['label_vec'], dtype=torch.float32)
            
            if self.transform:
                image = self.transform(image)
                
            return image, label
        except Exception as e:
            return torch.zeros((3, IMG_SIZE, IMG_SIZE)), torch.zeros(len(TARGET_CLASSES))

if os.path.exists(SUBSET_METADATA_PATH):
    df = pd.read_csv(SUBSET_METADATA_PATH)
    def encode(l): return [1 if c in str(l).split('|') else 0 for c in TARGET_CLASSES]
    df['label_vec'] = df['Finding Labels'].apply(encode)

    if os.path.exists(DATA_DIR):
        available = set(os.listdir(DATA_DIR))
        df = df[df['Image Index'].isin(available)].reset_index(drop=True)
        
    train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(NovelDataset(train_df, DATA_DIR, transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(NovelDataset(val_df, DATA_DIR, transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"DataLoaders Ready. Total images: {len(df)}")

DataLoaders Ready. Total images: 5500


## 2. Hybrid Model (CNN + ViT)

In [16]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        res = torch.cat([avg_out, max_out], dim=1)
        res = self.conv(res)
        return x * self.sigmoid(res)

class HybridCheXNet(nn.Module):
    def __init__(self, num_classes=15):
        super().__init__()
        cnn = models.densenet121(weights='IMAGENET1K_V1')
        self.cnn_features = cnn.features
        self.cnn_pool = nn.AdaptiveAvgPool2d(1)
        self.vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        self.spatial_att = SpatialAttention()
        self.classifier = nn.Sequential(
            nn.Linear(1024 + 768, 512),
            nn.ReLU()
        )
        self.final_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        c_feat = self.cnn_features(x)
        c_feat = self.spatial_att(c_feat)
        c_feat = self.cnn_pool(c_feat).view(x.size(0), -1) 
        v_feat = self.vit(x)
        combined = torch.cat([c_feat, v_feat], dim=1)
        out = self.classifier(combined)
        return self.final_fc(out)

model = HybridCheXNet().to(device)
print("Hybrid Model Ready.")

Hybrid Model Ready.


## 3. Training Config

In [17]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        return F_loss.mean()

criterion = FocalLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 4. Execution Loop

In [18]:
def train_novel_pipeline():
    if 'train_loader' not in globals(): return
    best_v_auc = 0
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
        
        scheduler.step()
        
        # Eval
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                out = torch.sigmoid(model(imgs))
                all_preds.append(out.cpu().numpy())
                all_labels.append(labels.numpy())
        
        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        
        try:
            auc = roc_auc_score(all_labels, all_preds, average='macro', multi_class='ovr')
            print(f"Epoch {epoch+1} Mean AUC: {auc:.4f}")
            if auc > best_v_auc:
                best_v_auc = auc
                torch.save(model.state_dict(), 'novel_subset_best.pth')
                print("--- Saved Best Novel Model ---")
        except Exception as e: print(f"AUC Error: {e}")

train_novel_pipeline()

Epoch 1:   0%|          | 0/293 [00:00<?, ?it/s]

Epoch 1 Mean AUC: 0.7037
--- Saved Best Novel Model ---


Epoch 2:   0%|          | 0/293 [00:00<?, ?it/s]

Epoch 2 Mean AUC: 0.7287
--- Saved Best Novel Model ---
